Шаг 1: Настройка окружения и импорт библиотек

In [13]:
# Установка зависимостей (для RAG, веб-скрапинга и Vosk)
!pip install openai pymupdf numpy requests beautifulsoup4 vosk pydub docling
# Скачиваем небольшую русскую модель Vosk
!wget -nc https://alphacephei.com/vosk/models/vosk-model-small-ru-0.22.zip
# Распаковываем и переименовываем папку для удобства
!unzip -n vosk-model-small-ru-0.22.zip
!mv vosk-model-small-ru-0.22 vosk-model-folder

# Импорт необходимых модулей
import fitz  # Главный модуль PyMuPDF
import os # Стандартный модуль для взаимодействия с операционной системой
import numpy as np # Библиотека для научных вычислений
import requests # Библиотека для выполнения HTTP-запросов
import json # Модуль для работы с данными в формате JSON
import wave # Стандартный модуль для чтения и записи файлов в формате WAV
from bs4 import BeautifulSoup # Основной класс библиотеки BeautifulSoup 4
from io import BytesIO # Класс из модуля io для работы с бинарными данными в памяти как с файлом
from openai import OpenAI # Клиентская библиотека для взаимодействия с OpenAI API
from vosk import Model, KaldiRecognizer # Класс для загрузки языковой модели Vosk и класс, который выполняет непосредственно распознавание речи
from pydub import AudioSegment # Класс для манипуляций с аудио (обрезка, конвертация, изменение параметров)
from google.colab import files # Для загрузки файлов
from docling.document_converter import DocumentConverter


print("✅ Зависимости установлены.")

File ‘vosk-model-small-ru-0.22.zip’ already there; not retrieving.

Archive:  vosk-model-small-ru-0.22.zip
   creating: vosk-model-small-ru-0.22/
   creating: vosk-model-small-ru-0.22/graph/
   creating: vosk-model-small-ru-0.22/graph/phones/
  inflating: vosk-model-small-ru-0.22/graph/phones/word_boundary.int  
  inflating: vosk-model-small-ru-0.22/graph/Gr.fst  
  inflating: vosk-model-small-ru-0.22/graph/HCLr.fst  
  inflating: vosk-model-small-ru-0.22/graph/disambig_tid.int  
   creating: vosk-model-small-ru-0.22/am/
  inflating: vosk-model-small-ru-0.22/am/final.mdl  
  inflating: vosk-model-small-ru-0.22/README  
   creating: vosk-model-small-ru-0.22/conf/
  inflating: vosk-model-small-ru-0.22/conf/model.conf  
  inflating: vosk-model-small-ru-0.22/conf/mfcc.conf  
   creating: vosk-model-small-ru-0.22/ivector/
  inflating: vosk-model-small-ru-0.22/ivector/final.dubm  
  inflating: vosk-model-small-ru-0.22/ivector/global_cmvn.stats  
  inflating: vosk-model-small-ru-0.22/ivector/

Шаг 2: Загрузка файла

In [2]:
# Блок загрузки для PDF
print("Начало загрузки файла...")
uploaded = files.upload()

# Сохраняем имя файла для дальнейшей работы

#Берем первый файл
pdf_path = list(uploaded.keys())[0]

print(f"✅ Файл '{pdf_path}' успешно загружен.")

Начало загрузки файла...


Saving instrukciya-samsung-galaxy-s23-ultra-6.pdf to instrukciya-samsung-galaxy-s23-ultra-6.pdf
✅ Файл 'instrukciya-samsung-galaxy-s23-ultra-6.pdf' успешно загружен.


Шаг 3: Предварительная обработка данных

In [3]:
def parse_pdf_document(file_path):
    """Извлекает и конкатенирует текст из каждой страницы PDF.

    :param file_path: Путь к PDF файлу.
    :return: Одна строка, содержащая весь текст документа.
    """
    converter = DocumentConverter()
    result = converter.convert(file_path)
    return result.document.export_to_markdown()

document_text = parse_pdf_document(pdf_path)
print(f"Извлечено {len(document_text)} символов.")

[INFO] 2025-12-03 10:26:13,744 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-12-03 10:26:13,750 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.4.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_infer.pth
[INFO] 2025-12-03 10:26:15,125 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2025-12-03 10:26:15,554 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2025-12-03 10:26:15,556 [RapidOCR] torch.py:54: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2025-12-03 10:26:15,730 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-12-03 10:26:15,732 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.4.0/torch/PP-OCRv4/cls/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2025-12-03 10:26:16,659 [RapidOCR] downlo

Извлечено 281037 символов.


3.2 Парсинг веб-страницы

In [14]:
web_text = ""

def parse_website_data(url):
  converter = DocumentConverter()
  result = converter.convert(url)
  return result.document.export_to_markdown()


In [15]:
# Запуск парсинга

WEBSITE_URL = "https://habr.com/ru/news/968856/"

print(f"-> Обработка ВЕБ-САЙТА: {WEBSITE_URL}")
web_text = parse_website_data(WEBSITE_URL)

print("\n" + "*"*40)
print(f"ИЗВЛЕЧЕННЫЙ ТЕКСТ (первые 800 символов):\n{web_text[:800]}...")
print("*"*40)

-> Обработка ВЕБ-САЙТА: https://habr.com/ru/news/968856/

****************************************
ИЗВЛЕЧЕННЫЙ ТЕКСТ (первые 800 символов):
# Figure AI завершила почти годичные испытания человекоподобных роботов Figure 02 на заводе BMW в Южной Каролине

Время на прочтение 2 мин Охват и читатели 5.4K [Робототехника](/ru/hubs/robot) [Транспорт](/ru/hubs/transport) [Будущее здесь](/ru/hubs/futurenow) [Развитие стартапа](/ru/hubs/startuprise)

Калифорнийский стартап Figure AI [отчитался](https://interestingengineering.com/ai-robotics/figure-humanoid-robots-retires-bmw) о завершении 11-месячных испытаний человекоподобных роботов Figure 02 (F.02) на заводе BMW в штате Южная Каролина. Стартап подвёл итоги развёртывания Figure 02 на фоне снятия модели с производства.

<!-- image -->

Пилотный проект был частью совместной работы компаний по испытанию гуманоидных роботов на настоящей сборочной линии. Figure AI [сообщила](https://x.com/F...
****************************************


3.3 Чанкинг (Разбиение) по Скользящему Окну

In [16]:
def create_sliding_window_chunks(document_text, max_chars=1000, stride_overlap=200):
    """Разбивает текст на фрагменты, используя скользящее окно."""
    corpus_fragments = []
    # Шаг, на который сдвигается окно (Размер - Перекрытие)
    stride = max_chars - stride_overlap

    if stride <= 0:
        raise ValueError("Размер перекрытия должен быть меньше размера фрагмента.")

    # Создаём последовательность начальных позиций для разрезания текста с шагом stride.
    for start_index in range(0, len(document_text), stride):
        # Окно заканчивается на start_index + max_chars
        end_index = min(start_index + max_chars, len(document_text))
        fragment = document_text[start_index:end_index]
        corpus_fragments.append(fragment)

        # Если конец фрагмента совпал с концом текста, останавливаемся
        if end_index == len(document_text):
            break

    return corpus_fragments

3.4 Применяем функцию для данных полученных из разных источников

In [5]:
# PDF-документ
corpus_fragments_pdf = create_sliding_window_chunks(document_text, max_chars=1000, stride_overlap=200)
print(f"Документ разбит на {len(corpus_fragments_pdf)} фрагментов для индексации.")

Документ разбит на 352 фрагментов для индексации.


In [17]:
# Сайт
corpus_fragments_web = create_sliding_window_chunks(web_text, max_chars=250, stride_overlap=50)
print(f"Документ разбит на {len(corpus_fragments_web)} фрагментов для индексации.")

Документ разбит на 30 фрагментов для индексации.


Шаг 4: Индексация (Векторизация)

4.1 Эмбеддинги и Векторное Пространство



In [18]:
# 🔑 Установка API ключа. Получите его в Nebius Studio.
os.environ["OPENAI_API_KEY"] = "v1.CmQKHHN0YXRpY2tleS1lMDBia2plZ2Q4ZTJyYjUxM2ESIXNlcnZpY2VhY2NvdW50LWUwMHJrcHhkMTJoaGU1YWp3ZzIMCKz_nMkGEK-F14oDOgwIrYK1lAcQwML3xAFAAloDZTAw.AAAAAAAAAAFmAHfiJC4n9CgfRlsMLkNXor2vbUbKr5lf0RV4NrtkXvFpNRiIldreTI9bSPaRVASr8fwv5I_clks8wD3ypDYL"

# Настройка клиента для Nebius AI (совместим с OpenAI API)
client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.getenv("OPENAI_API_KEY")
)

def create_embeddings(text_list, model="BAAI/bge-multilingual-gemma2"):
    """Вызывает API для преобразования списка текстов в список векторов (эмбеддингов)."""
    # Вызов метода embeddings.create для получения векторов
    response = client.embeddings.create(input=text_list, model=model)
    return response.data

In [8]:
# PDF-документ
# Индексация: Генерация векторов для всех фрагментов корпуса
print("⏳ Генерация эмбеддингов для всего корпуса... (требует времени)")
fragment_embeddings_pdf = create_embeddings(corpus_fragments_pdf)
print(f"✅ Индексация завершена. Получено {len(fragment_embeddings_pdf)} векторов.")

⏳ Генерация эмбеддингов для всего корпуса... (требует времени)
✅ Индексация завершена. Получено 352 векторов.


In [19]:
# Сайт
# Индексация: Генерация векторов для всех фрагментов корпуса
print("⏳ Генерация эмбеддингов для всего корпуса... (требует времени)")
fragment_embeddings_web = create_embeddings(corpus_fragments_web)
print(f"✅ Индексация завершена. Получено {len(fragment_embeddings_web)} векторов.")

⏳ Генерация эмбеддингов для всего корпуса... (требует времени)
✅ Индексация завершена. Получено 30 векторов.


Шаг 5: Извлечение контекста (Retrieval)

In [20]:
def calculate_cosine_similarity(vec1, vec2):
    """Вычисляет косинусное сходство между двумя векторами NumPy."""
    # np.dot(vec1, vec2) - скалярное произведение
    # np.linalg.norm(vec) - Евклидова норма (длина) вектора
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def retrieve_context(user_query, corpus_fragments, fragment_embeddings, k_neighbors=3):
    """Находит топ-K фрагментов корпуса, наиболее близких к запросу по смыслу.

    :param k_neighbors: Количество самых релевантных фрагментов для извлечения.
    :return: Список из K фрагментов текста.
    """
    # 1. Векторизация запроса
    query_embedding_response = client.embeddings.create(
        input=[user_query],
        model="BAAI/bge-multilingual-gemma2"
    )
    # Преобразуем список чисел в вектор NumPy для расчетов
    query_embedding = np.array(query_embedding_response.data[0].embedding)

    # 2. Расчет сходства
    similarity_scores = []
    for i, emb_data in enumerate(fragment_embeddings):
        fragment_vector = np.array(emb_data.embedding)
        score = calculate_cosine_similarity(query_embedding, fragment_vector)
        similarity_scores.append((i, score))

    # 3. Сортировка и выбор топ-K
    # Сортируем по убыванию (чем выше score, тем лучше)
    similarity_scores.sort(key=lambda x: x[1], reverse=True)
    top_k_indices = [index for index, score in similarity_scores[:k_neighbors]]

    return [corpus_fragments[i] for i in top_k_indices]

Шаг 6: Генерация ответа (Augmented Generation)

In [21]:
def generate_augmented_response(query, context_fragments, model_name="Qwen/Qwen3-32B"):
    """Формирует промпт из контекста и запроса и вызывает LLM для ответа."""

    system_prompt = "Вы — экспертный ассистент по работе с документами. Ваш ответ должен быть строго основан на предоставленном КОНТЕКСТЕ. Если в КОНТЕКСТЕ нет информации для полного ответа, вы должны сообщить, что информация не найдена, не выдумывая фактов."

    # Объединяем найденные фрагменты в единый, легко читаемый контекст
    context_text = "\n\n--- РЕЛЕВАНТНЫЙ ФРАГМЕНТ ---\n\n".join(context_fragments)

    full_user_message = f"""
    КОНТЕКСТ ДЛЯ ОТВЕТА:
    {context_text}

    ВОПРОС ПОЛЬЗОВАТЕЛЯ:
    {query}
    """

    response = client.chat.completions.create(
        model=model_name,
        temperature=0, # Обеспечивает фактическую точность и снижает "галлюцинации"
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": full_user_message}
        ]
    )

    return response.choices[0].message.content

Шаг 7: Тестирование системы

In [22]:
# Сначала можем посмотреть как LLM ответит на наш запрос без работы RAG.
# Пример запроса "Что ты знаешь про Figure 02?"
# Интерактивный ввод вопроса
user_query = input("Введите ваш вопрос: ")

# Генерируем финальный ответ
response = client.chat.completions.create(
    model="Qwen/Qwen3-32B",
    temperature=0, # Обеспечивает фактическую точность и снижает "галлюцинации"
    messages=[
        {"role": "user", "content": user_query}
    ]
)

final_answer = response.choices[0].message.content

print("\n" + "="*60)
print("ФИНАЛЬНЫЙ ОТВЕТ СИСТЕМЫ RAG:")
print("="*60)
print(final_answer)
print("="*60)

Введите ваш вопрос: Что ты знаешь о Figure 02?

ФИНАЛЬНЫЙ ОТВЕТ СИСТЕМЫ RAG:
<think>
Хорошо, пользователь спрашивает о Figure 02. Сначала мне нужно понять, о чем именно идет речь. Figure 02 может быть названием проекта, группы, альбома или чего-то еще. Поскольку я не уверен, стоит проверить несколько возможных интерпретаций.

Первое, что приходит в голову, это музыкальная группа или исполнитель. Возможно, это арт-проект или коллектив, связанный с электронной музыкой, учитывая, что названия вроде Figure 02 часто используются в этом жанре. Нужно поискать информацию о группах или артистах с таким названием. Если не найду, стоит рассмотреть другие варианты.

Также Figure 02 может быть частью какого-то альбома или трека. Например, в музыке часто используются нумерованные названия для треков или серий. Возможно, это второй трек в серии, обозначенной как Figure. Нужно проверить музыкальные базы данных или каталоги, такие как Spotify, Apple Music или AllMusic, чтобы найти соответствующие запис

In [12]:
# PDF
# Пример запроса "Как открыть камеру?"
# Интерактивный ввод вопроса
user_query = input("Введите ваш вопрос по загруженному файлу: ")

print("\n" + "*"*40)
print("🔍 Этап 1: Извлечение контекста (Retrieval)")

# Извлекаем 3 самых релевантных фрагмента
top_k_fragments = retrieve_context(user_query, corpus_fragments_pdf, fragment_embeddings_pdf, k_neighbors=3)

# Выводим найденные фрагменты для проверки прозрачности работы
for i, chunk in enumerate(top_k_fragments):
    print(f"\n--- Найденный фрагмент {i+1} ---\n{chunk[:200]}...") # Выводим только первые 200 символов

print("\n" + "*"*40)
print("🤖 Этап 2: Генерация ответа (Augmentation)")

# Генерируем финальный ответ
final_answer = generate_augmented_response(user_query, top_k_fragments)

print("\n" + "="*60)
print("ФИНАЛЬНЫЙ ОТВЕТ СИСТЕМЫ RAG:")
print("="*60)
print(final_answer)
print("="*60)

Введите ваш вопрос по загруженному файлу: Что такое камера?

****************************************
🔍 Этап 1: Извлечение контекста (Retrieval)

--- Найденный фрагмент 1 ---
тии камеры.
- Вибрационная обратная связь : вибрация устройства в определенных ситуациях, например, при касании кнопки камеры.

<!-- image -->

## Конфиденциальность

- Уведомление о конфиденциальност...

--- Найденный фрагмент 2 ---
к сообщений

Откройте приложение Сообщения , нажмите кнопку → Настройки . Можно заблокировать нежелательные сообщения, изменить настройки уведомлений и других функций.

## Камера

## Введение

Фото- и...

--- Найденный фрагмент 3 ---
>

<!-- image -->

## Просмотр сообщений

- 1 Откройте приложение Сообщения и выберите пункт Разговоры .
- 2 В списке сообщений выберите контакт или номер телефона.
- Чтобы ответить на сообщение, косн...

****************************************
🤖 Этап 2: Генерация ответа (Augmentation)

ФИНАЛЬНЫЙ ОТВЕТ СИСТЕМЫ RAG:
<think>
Хорошо, пользователь спрашивает

In [23]:
# Сайт
# Пример запроса "Что ты знаешь про Figure 02?"
# Интерактивный ввод вопроса
user_query = input("Введите ваш вопрос по загруженному файлу: ")

print("\n" + "*"*40)
print("🔍 Этап 1: Извлечение контекста (Retrieval)")

# Извлекаем 3 самых релевантных фрагмента
top_k_fragments = retrieve_context(user_query, corpus_fragments_web, fragment_embeddings_web, k_neighbors=1)

# Выводим найденные фрагменты для проверки прозрачности работы
for i, chunk in enumerate(top_k_fragments):
    print(f"\n--- Найденный фрагмент {i+1} ---\n{chunk[:200]}...") # Выводим только первые 200 символов

print("\n" + "*"*40)
print("🤖 Этап 2: Генерация ответа (Augmentation)")

# Генерируем финальный ответ
final_answer = generate_augmented_response(user_query, top_k_fragments)

print("\n" + "="*60)
print("ФИНАЛЬНЫЙ ОТВЕТ СИСТЕМЫ RAG:")
print("="*60)
print(final_answer)
print("="*60)

Введите ваш вопрос по загруженному файлу: Что ты знаешь о Figure 02?

****************************************
🔍 Этап 1: Извлечение контекста (Retrieval)

--- Найденный фрагмент 1 ---
_type=posts&order=relevance&q=[figure])
- [f.02](/ru/search/?target_type=posts&order=relevance&q=[f.02])
- [figure 02](/ru/search/?target_type=posts&order=relevance&q=[figure+02])
- [роботы](/ru/searc...

****************************************
🤖 Этап 2: Генерация ответа (Augmentation)

ФИНАЛЬНЫЙ ОТВЕТ СИСТЕМЫ RAG:
<think>
Хорошо, пользователь спрашивает о Figure 02. Нужно проверить предоставленный контекст. Вижу, что в контексте есть ссылки на [f.02], [figure 02] и [роботы]. Однако самих описаний или информации о Figure 02 нет. Все ссылки ведут на поисковые запросы, но содержимое не указано. Значит, в контексте нет конкретных данных о Figure 02. Следует ответить, что информация не найдена.
</think>

На основе предоставленного контекста информация о Figure 02 отсутствует. В тексте указаны только поисковы